In [ ]:
from pyspark.sql import functions as F

# TODO: configurar la ruta real del bucket S3.
# Ejemplo de estructura esperada:
# s3://<bucket>/<prefijo>/orders.csv
# s3://<bucket>/<prefijo>/products.csv
RAW_PATH = "s3://REEMPLAZAR_BUCKET/raw/"

CATALOG = "workspace"
SCHEMA = "retail_analytics"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

In [ ]:
# Cargar dos datasets de entrada.
# Ajusta los nombres de archivo según los datasets elegidos.
orders_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(RAW_PATH + "orders.csv")
)

products_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(RAW_PATH + "products.csv")
)

display(orders_raw.limit(10))
display(products_raw.limit(10))

In [ ]:
# Persistir Bronze en Delta.
# La ingesta inicial conserva columnas tal como fueron leídas.
(
    orders_raw.write.format("delta").mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_orders")
)

(
    products_raw.write.format("delta").mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_products")
)

print("Orders Bronze:", spark.table(f"{CATALOG}.{SCHEMA}.bronze_orders").count())
print("Products Bronze:", spark.table(f"{CATALOG}.{SCHEMA}.bronze_products").count())